In [17]:
import json
import numpy as np
import pandas as pd
import chromadb
from pathlib import Path
from chromadb.utils import embedding_functions
from llama_cpp import Llama, LlamaGrammar

# DARPA pipeline — outputs go to a separate subdirectory.
RESULTS_DIR  = Path("../data/results/darpa")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SHARED_DIR   = Path("../data/results")
ATTCK_DIR   = Path("../data/attck")
CHROMA_DIR  = Path("../data/chroma")
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH     = "../models/qwen2.5-3b-instruct-q4_k_m.gguf"
COMMUNITY_FILE = SHARED_DIR  / "community_assignments.csv"
TRIPLES_FILE   = SHARED_DIR  / "community_triples.json"
STIX_FILE      = ATTCK_DIR   / "enterprise-attack.json"
GRAPH_NODES    = RESULTS_DIR / "knowledge_graph_nodes.csv"  # written by darpa_3
GRAPH_EDGES    = RESULTS_DIR / "knowledge_graph_edges.csv"  # written by darpa_3
REPORTS_FILE   = RESULTS_DIR / "darpa_rag_reports.csv"
METRICS_FILE   = RESULTS_DIR / "darpa_rag_metrics.json"

EMBED_MODEL       = "all-MiniLM-L6-v2"
TOP_K             = 5
RANDOM_SEED       = 42
MAX_CHARS_PER_DOC = 400

print("Environment ready.")

Environment ready.


In [18]:
# Load all upstream outputs
print('Resolved COMMUNITY_FILE:', COMMUNITY_FILE.resolve())
if not COMMUNITY_FILE.exists():
    print('ERROR: COMMUNITY_FILE not found:', COMMUNITY_FILE.resolve())
    print('Ensure upstream notebooks wrote ../data/results/community_assignments.csv and re-run stage1/stage2.' )
    raise FileNotFoundError('COMMUNITY_FILE not found: ' + str(COMMUNITY_FILE.resolve()))
community_df = pd.read_csv(COMMUNITY_FILE, low_memory=False)

with open(TRIPLES_FILE, "r", encoding="utf-8") as f:
    community_triples = json.load(f)

kg_edges = pd.read_csv(GRAPH_EDGES)
kg_nodes = pd.read_csv(GRAPH_NODES)

# Diagnostic: show actual columns and unique values so mismatches are visible
print("community_df columns:", community_df.columns.tolist())
print("community_df shape:  ", community_df.shape)

# Detect the label column name defensively (DARPA = "label", CIC-IDS = "Label")
if "label" in community_df.columns:
    label_col = "label"
elif "Label" in community_df.columns:
    label_col = "Label"
else:
    raise ValueError(f"No label column found. Available: {community_df.columns.tolist()}")

print(f"Using label column: '{label_col}'")
print(f"Unique values: {community_df[label_col].unique()[:10]}")

# Detect the tactic column name defensively
tactic_col = "attck_tactic" if "attck_tactic" in community_df.columns else None
if tactic_col:
    print(f"Tactic distribution:\n{community_df[tactic_col].value_counts().to_string()}")

# Identify attack communities by dominant label per community
# A community is an attack community if the majority of its events are labelled ATTACK.
attack_labels = community_df.groupby("community_id")[label_col].agg(
    lambda x: x.value_counts().index[0]
)
print(f"\nDominant label per community:\n{attack_labels.value_counts().to_string()}")

# Accept both "ATTACK" and any non-BENIGN technique ID (covers CIC-IDS reuse)
attack_cids = attack_labels[
    ~attack_labels.isin(["BENIGN", "Benign"])
].index.tolist()

eval_df   = community_df[community_df["community_id"].isin(attack_cids)].copy()
eval_cids = sorted(attack_cids)

print(f"\nLoaded {len(community_df)} alerts across {community_df['community_id'].nunique()} communities")
print(f"Attack communities to evaluate: {len(eval_cids)}")
print(f"Knowledge graph: {len(kg_nodes)} nodes, {len(kg_edges)} edges")

if len(eval_cids) == 0:
    print("\nWARNING: No attack communities found.")
    print("This usually means the label column contains only BENIGN values,")
    print("which means notebook 1 produced no attack-labelled events.")
    print("Check that ATTACKER_IPS and ATTACK_WINDOWS matched events in your data.")


Resolved COMMUNITY_FILE: /home/amenadiel/RAG-Based-Incident-Reporting-on-SIEM-Log-Streams/data/results/community_assignments.csv
community_df columns: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Flow Bytes/s', 'SYN Flag Count', 'RST Flag Count', 'ACK Flag Count', 'Label', 'attck_technique_id', 'attck_technique_name', 'attck_tactic', 'day', 'alert_text', 'alert_id', 'sample_id.1', 'community_id']
community_df shape:   (10684, 17)
Using label column: 'Label'
Unique values: <StringArray>
[        'Web Attack XSS',                    'Bot',            'FTP Patator',
 'Web Attack Brute Force',               'DoS Hulk',               'PortScan',
                 'BENIGN',            'SSH Patator',                   'DDoS',
             'Heartbleed']
Length: 10, dtype: str
Tactic distribution:
attck_tactic
Command And Control    2000
Credential Access      2000
Impact                 2000
Discovery              2000
Benign                 2000
Executi

In [19]:
# Initialize ChromaDB and populate with ATT&CK technique descriptions
# The same embedding model is used here as in Stage 1 to ensure that
# similarity comparisons are made within a consistent vector space.
ef         = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
client     = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection(name="attack_techniques", embedding_function=ef)

if collection.count() == 0:
    print("Populating ChromaDB...")
    with open(STIX_FILE, "r", encoding="utf-8") as f:
        bundle = json.load(f)

    docs, ids, metas = [], [], []
    for obj in bundle.get("objects", []):
        if obj.get("type") != "attack-pattern" or obj.get("revoked"):
            continue
        tid = None
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                tid = ref.get("external_id")
                break
        if not tid:
            continue
        tactics = [
            p["phase_name"].replace("-", " ").title()
            for p in obj.get("kill_chain_phases", [])
            if p.get("kill_chain_name") == "mitre-attack"
        ]
        text = (
            f"ID: {tid}\n"
            f"Name: {obj.get('name', '')}\n"
            f"Tactic: {', '.join(tactics)}\n"
            f"Description: {obj.get('description', '')}"
        )
        docs.append(text)
        ids.append(tid)
        metas.append({"technique_id": tid, "name": obj.get("name", ""), "tactic": ", ".join(tactics)})

    collection.add(ids=ids, documents=docs, metadatas=metas)
    print(f"Inserted {collection.count()} ATT&CK technique descriptions")
else:
    print(f"ChromaDB ready: {collection.count()} technique descriptions")

ChromaDB ready: 703 technique descriptions


In [20]:
# LLM initialization
print(f"Loading {MODEL_PATH}...")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096,
    n_gpu_layers=-1,
    n_threads=8,
    n_batch=256,
    verbose=False,
    seed=RANDOM_SEED,
)
print("LLM ready.")

Loading ../models/qwen2.5-3b-instruct-q4_k_m.gguf...


llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


LLM ready.


In [21]:
# Hypothetical Document Embedding (HyDE) query builder
#
# DESIGN RATIONALE
# ----------------
# The previous version built the retrieval query by concatenating raw triples
# into short sentences (e.g. "ftp client scans ftp port 21 service."). These
# short strings sit far from the long ATT&CK technique descriptions in the
# embedding space, which is why the retrieval hit rate was only ~14%.
#
# HyDE solves this by asking the LLM to generate a hypothetical ATT&CK-style
# description of the observed behaviour *before* querying ChromaDB. Because the
# hypothetical description uses the same vocabulary and length as real technique
# descriptions, it sits much closer to them in embedding space.
#
# WHY THIS IS NOT CHEATING
# ------------------------
# The LLM generates the hypothetical description from the extracted triples
# alone, with no knowledge of the ground-truth technique ID. It is making an
# educated guess based on general knowledge, and that guess is then used only
# as a retrieval query — the final technique is still selected by semantic
# similarity from the vector store.
#
# Graph traversal context:
# We also look up the 3 highest-weight edges in the knowledge graph whose
# nodes overlap with this community. These recurring patterns across multiple
# communities are included in the HyDE prompt to give the model richer context
# than the community triples alone provide.

def build_hyde_query(cid: int) -> tuple:
    """Return (hyde_query_text, raw_triple_text, alert_context_text).

    hyde_query_text   : LLM-generated hypothetical ATT&CK description for ChromaDB
    raw_triple_text   : flat triple text retained for the report prompt
    alert_context_text: raw alert snippets retained for the report prompt
    """
    group   = community_df[community_df["community_id"] == cid]
    triples = community_triples.get(str(cid), [])

    # Build a plain-language summary of the triples
    triple_sentences = []
    for t in triples:
        subj = t.get("subject",  "").replace("_", " ")
        rel  = t.get("relation", "").replace("_", " ").lower()
        tgt  = t.get("target",   "").replace("_", " ")
        if subj and rel and tgt:
            triple_sentences.append(f"{subj} {rel} {tgt}")
    raw_triple_text = ". ".join(triple_sentences)

    # Pull high-weight graph edges involving this community's entities
    # to give the HyDE prompt structural context beyond the 4 triples
    community_entities = set()
    for t in triples:
        community_entities.add(t.get("subject", ""))
        community_entities.add(t.get("target",  ""))

    related_edges = kg_edges[
        (kg_edges["source"].isin(community_entities) |
         kg_edges["target"].isin(community_entities)) &
        (kg_edges["layer"] == "behavioral")
    ].sort_values("weight", ascending=False).head(3)

    graph_context_parts = []
    if not related_edges.empty:
        for _, row in related_edges.iterrows():
            s = row["source"].replace("_", " ")
            r = row["relation"].replace("_", " ").lower()
            o = row["target"].replace("_", " ")
            graph_context_parts.append(f"{s} {r} {o}")
    graph_context = "; ".join(graph_context_parts)

    # HyDE prompt: ask the LLM to write a description that sounds like
    # a MITRE ATT&CK technique entry, grounded in the observed triples
    hyde_prompt = f"""[INST] You are a cybersecurity threat analyst.
Based on the observed network behaviours below, write a concise technical description
of the attack technique being used. Write in the style of a MITRE ATT&CK technique
description: describe what the adversary does, which resources they target, and what
the observable indicators are. Do NOT name a specific technique ID.

OBSERVED BEHAVIOURS:
{raw_triple_text}

RECURRING GRAPH PATTERNS:
{graph_context or 'None identified'}

Write 3 to 5 sentences. [/INST]"""

    out = llm(
        hyde_prompt,
        max_tokens=200,
        temperature=0.2,   # slight temperature allows richer vocabulary than 0
        seed=RANDOM_SEED,
        repeat_penalty=1.1,
        stop=["[/INST]"]
    )
    hyde_query = out["choices"][0]["text"].strip()

    alert_context = " ".join(group["alert_text"].dropna().head(3).tolist())
    return hyde_query, raw_triple_text, alert_context


# Inspect the HyDE output for the first attack community
if not eval_cids:
    print("No attack communities found to inspect; skipping HyDE example.")
else:
    print(f"Example HyDE query for community {eval_cids[0]}:")
    hq, rt, _ = build_hyde_query(eval_cids[0])
    print(hq)

Example HyDE query for community 0:
Based on the observed network behaviours, a sophisticated adversary is employing a multi-stage attack technique. Initially, an FTP brute force client attempts to gain unauthorized access by attempting credential-based authentication against an FTP service. Concurrently, an HTTP scanner performs reconnaissance activities to identify and map out potential targets within the internal network. Following this initial phase of reconnaissance, the HTTP scanner establishes command and control (C2) channels for further operations. Finally, the HTTP scanner exfiltrates sensitive data from internal data stores to external locations. Observable indicators include failed FTP login attempts, unusual outbound HTTP traffic patterns, C2 channel communications, and anomalous data exfiltration activity.


In [22]:
# Report schema
# The grammar constrains the LLM output to a fixed JSON structure
# so that report parsing is deterministic and does not require error-prone
# post-processing of free-form text.
REPORT_SCHEMA = {
    "type": "object",
    "properties": {
        "technique_id": {"type": "string"},
        "tactic":       {"type": "string"},
        "summary":      {"type": "string"},
        "evidence":     {"type": "string"},
        "next_step":    {"type": "string"}
    },
    "required": ["technique_id", "tactic", "summary", "evidence", "next_step"]
}
report_grammar = LlamaGrammar.from_json_schema(json.dumps(REPORT_SCHEMA))
print("Report grammar initialized.")

Report grammar initialized.


In [23]:
# Report generation functions
#
# RAG REPORT
# Uses the HyDE query to retrieve relevant ATT&CK techniques from ChromaDB,
# then passes the retrieved context to the LLM for grounded report generation.
# The allowed_ids constraint prevents the LLM from hallucinating technique IDs
# that were not retrieved.
#
# BASELINE REPORT
# Uses only the raw triple text with no retrieval. This is the comparison
# condition for the thesis evaluation: it demonstrates what the LLM produces
# from the graph representation alone, without any external knowledge grounding.

def generate_rag_report(
    hyde_query:    str,
    raw_triples:   str,
    alert_context: str,
    retrieved_docs: list,
    retrieved_metas: list
) -> dict:
    allowed_ids = [m.get("technique_id", "") for m in retrieved_metas]
    allowed_str = ", ".join(allowed_ids)
    context_block = "\n\n".join([
        f"[{i+1}] {str(doc)[:MAX_CHARS_PER_DOC]}"
        for i, doc in enumerate(retrieved_docs)
    ])

    prompt = f"""[INST] You are a cybersecurity analyst. Write a concise incident report.

Choose technique_id from this list only: {allowed_str}

OBSERVED BEHAVIOUR (extracted graph triples):
{raw_triples}

RAW ALERT CONTEXT:
{alert_context}

RETRIEVED ATT&CK TECHNIQUES:
{context_block}

Rules:
- summary  : one sentence describing what the attacker did
- evidence : two or three specific observations from the alert context that support your choice
- next_step: one concrete analyst action

Return ONLY valid JSON. [/INST]"""

    out = llm(
        prompt,
        max_tokens=400,
        temperature=0,
        seed=RANDOM_SEED,
        grammar=report_grammar,
        repeat_penalty=1.1
    )
    raw = out["choices"][0]["text"].strip()

    try:
        result = json.loads(raw)
        # Reject hallucinated IDs: the LLM must select from the retrieved set
        if result.get("technique_id", "") not in allowed_ids:
            result["technique_id"] = "Unknown"
            result["evidence"]     = "[Hallucinated ID corrected to Unknown]"
        return result
    except json.JSONDecodeError:
        return {
            "technique_id": "Unknown", "tactic": "Unknown",
            "summary": "Parse error", "evidence": "N/A", "next_step": "Manual review"
        }


def generate_baseline_report(raw_triples: str) -> dict:
    """Generate a report from graph triples alone, with no retrieval context.
    This is the comparison condition for the thesis evaluation."""
    prompt = f"""[INST] You are a cybersecurity analyst. Write a concise incident report.

OBSERVED BEHAVIOUR (extracted graph triples):
{raw_triples}

Rules:
- summary  : one sentence describing what the attacker did
- evidence : two or three specific observations that support your choice
- next_step: one concrete analyst action

Identify the most likely MITRE ATT&CK technique and return ONLY valid JSON. [/INST]"""

    out = llm(
        prompt,
        max_tokens=400,
        temperature=0,
        seed=RANDOM_SEED,
        grammar=report_grammar,
        repeat_penalty=1.1
    )
    raw = out["choices"][0]["text"].strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {
            "technique_id": "Unknown", "tactic": "Unknown",
            "summary": "Parse error", "evidence": "N/A", "next_step": "Manual review"
        }

In [24]:
# Evaluation loop
#
# For each attack community:
#   1. Build a HyDE query from the extracted triples and graph context
#   2. Retrieve the top-K ATT&CK techniques from ChromaDB using the HyDE query
#   3. Generate a RAG report grounded in the retrieved techniques
#   4. Generate a baseline report from triples alone (no retrieval)
#   5. Evaluate both against the ground-truth technique ID
#
# MATCH CRITERIA
# exact_match  : generated ID == ground truth ID
# parent_match : generated ID shares the same parent technique (e.g. T1110 == T1110.001)
# retrieval_hit: ground truth ID appears in the top-K retrieved set

def parent_match(gt_id: str, gen_id: str) -> bool:
    """True if both IDs share the same top-level technique (e.g. T1110.001 and T1110.003)."""
    if not gt_id or not gen_id or gen_id == "Unknown":
        return False
    return gt_id.split(".")[0] == gen_id.split(".")[0]


print("Warming up LLM")
_ = llm("[INST] Test. [/INST]", max_tokens=5, temperature=0, seed=RANDOM_SEED)
print("Starting evaluation\n")

results = []

for cid in eval_cids:
    group        = community_df[community_df["community_id"] == cid]
    # Use attck_technique_id if available, otherwise fall back to label column
    if "attck_technique_id" in group.columns:
        ground_truth = group["attck_technique_id"].mode().iloc[0]
    else:
        ground_truth = group[label_col].mode().iloc[0]

    # Step 1-2: HyDE query + retrieval
    hyde_query, raw_triples, alert_context = build_hyde_query(cid)
    retrieval     = collection.query(query_texts=[hyde_query], n_results=TOP_K)
    docs          = retrieval["documents"][0]
    metas         = retrieval["metadatas"][0]
    retrieved_ids = [m.get("technique_id", "") for m in metas]

    # Step 3: RAG report
    rag_report = generate_rag_report(hyde_query, raw_triples, alert_context, docs, metas)
    rag_id     = rag_report.get("technique_id", "Unknown")

    # Step 4: Baseline report
    base_report = generate_baseline_report(raw_triples)
    base_id     = base_report.get("technique_id", "Unknown")

    # Step 5: Evaluation
    retrieval_hit = ground_truth in retrieved_ids
    rag_grounded  = rag_id in retrieved_ids or rag_id == "Unknown"
    rag_exact     = (rag_id == ground_truth) and rag_grounded
    rag_parent    = parent_match(ground_truth, rag_id) and rag_grounded
    base_exact    = base_id == ground_truth
    base_parent   = parent_match(ground_truth, base_id)

    results.append({
        "community_id":       cid,
        "ground_truth":       ground_truth,
        "dominant_label":     group[label_col].mode().iloc[0],
        "dominant_tactic":    group[tactic_col].mode().iloc[0] if tactic_col else "Unknown",
        "hyde_query":         hyde_query,
        "retrieved_ids":      retrieved_ids,
        "retrieval_hit":      retrieval_hit,
        "rag_technique_id":   rag_id,
        "rag_grounded":       rag_grounded,
        "rag_exact_match":    rag_exact,
        "rag_parent_match":   rag_parent,
        "rag_tactic":         rag_report.get("tactic",    ""),
        "rag_summary":        rag_report.get("summary",   ""),
        "rag_evidence":       rag_report.get("evidence",  ""),
        "rag_next_step":      rag_report.get("next_step", ""),
        "base_technique_id":  base_id,
        "base_exact_match":   base_exact,
        "base_parent_match":  base_parent,
        "base_tactic":        base_report.get("tactic",   ""),
        "base_summary":       base_report.get("summary",  ""),
    })

    print(f"  Community {cid} [{ground_truth}] | "
          f"RAG: {rag_id} (exact={rag_exact}, parent={rag_parent}) | "
          f"Baseline: {base_id} (exact={base_exact}, parent={base_parent})")

print(f"\nEvaluation complete: {len(results)} communities")

Warming up LLM


Starting evaluation

  Community 0 [T1046] | RAG: T1071.002 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 1 [T1110.001] | RAG: T1563.001 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 4 [T1110.001] | RAG: T1056.003 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 5 [T1110.001] | RAG: T1071.004 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 11 [T1110.001] | RAG: T1071.001 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 12 [T1071.001] | RAG: T1102.002 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 13 [T1110.001] | RAG: T1110.003 (exact=False, parent=True) | Baseline: T1078 (exact=False, parent=False)
  Community 15 [T1046] | RAG: T1210 (exact=False, parent=False) | Baseline: T1078 (exact=False, parent=False)
  Community 17 [T1046] | RAG: T1573 (exact=False, pa

In [25]:
# Aggregate metrics
# Delta values measure the gain of the full GraphRAG pipeline over the baseline.
# A positive delta on retrieval_hit_rate confirms that HyDE is retrieving
# the correct technique more often than the raw triple query.
# A positive delta on parent_match confirms that graph-grounded generation
# at minimum identifies the correct technique family.
n = len(results)

metrics = {
    "model":                       "qwen2.5-3b-instruct-q4_k_m",
    "retrieval_strategy":          "HyDE (Hypothetical Document Embedding)",
    "n_communities_evaluated":     n,
    "top_k_retrieval":             TOP_K,
    "retrieval_hit_rate":          round(sum(r["retrieval_hit"]    for r in results) / n, 4),
    "rag_grounding_rate":          round(sum(r["rag_grounded"]     for r in results) / n, 4),
    "rag_exact_match_rate":        round(sum(r["rag_exact_match"]  for r in results) / n, 4),
    "rag_parent_match_rate":       round(sum(r["rag_parent_match"] for r in results) / n, 4),
    "baseline_exact_match_rate":   round(sum(r["base_exact_match"] for r in results) / n, 4),
    "baseline_parent_match_rate":  round(sum(r["base_parent_match"]for r in results) / n, 4),
}

metrics["delta_exact_match"]  = round(metrics["rag_exact_match_rate"]  - metrics["baseline_exact_match_rate"],  4)
metrics["delta_parent_match"] = round(metrics["rag_parent_match_rate"] - metrics["baseline_parent_match_rate"], 4)

print("METRICS")
print(json.dumps(metrics, indent=2))

print(f"\nRAG exact:       {metrics['rag_exact_match_rate']:.2%}")
print(f"Baseline exact:  {metrics['baseline_exact_match_rate']:.2%}")
print(f"Delta:           {metrics['delta_exact_match']:+.2%}")
print(f"\nRAG parent:      {metrics['rag_parent_match_rate']:.2%}")
print(f"Baseline parent: {metrics['baseline_parent_match_rate']:.2%}")
print(f"Delta:           {metrics['delta_parent_match']:+.2%}")
print(f"\nRetrieval hit:   {metrics['retrieval_hit_rate']:.2%}")

METRICS
{
  "model": "qwen2.5-3b-instruct-q4_k_m",
  "retrieval_strategy": "HyDE (Hypothetical Document Embedding)",
  "n_communities_evaluated": 10,
  "top_k_retrieval": 5,
  "retrieval_hit_rate": 0.3,
  "rag_grounding_rate": 1.0,
  "rag_exact_match_rate": 0.0,
  "rag_parent_match_rate": 0.1,
  "baseline_exact_match_rate": 0.0,
  "baseline_parent_match_rate": 0.0,
  "delta_exact_match": 0.0,
  "delta_parent_match": 0.1
}

RAG exact:       0.00%
Baseline exact:  0.00%
Delta:           +0.00%

RAG parent:      10.00%
Baseline parent: 0.00%
Delta:           +10.00%

Retrieval hit:   30.00%
